# Stage 00 — Universe and High-Recall Reproducibility

This notebook is a **read-only evidence walkthrough** for the governed Stage 00 chain: acquisition lineage → issuer exclusions → Item 1 packets → high-recall screen state. It is not a second implementation of the pipeline. Production logic remains in `src/` and `pipelines/`; immutable artifacts remain under `data/runs/`.

Default mode is `verify`: no SEC request, model call, credential resolution, run-directory creation, or artifact write occurs. Nothing here stages, commits, or edits a repository or data file.

Current state, as of ADR-138: the Stage 00 corpus is complete and hash-verified, and **an authoritative high-recall SCREEN release now exists** — `universe-screen-release-v1-20260823`, covering 7,042 packets — together with the human-review overlay and the immutable 4,045-row classifier candidate cohort built from them. Sections 6 and 7 still verify the named failed parent run, because that run's archive is what the governed continuation reused and its receipt remains the evidence for that reuse; it is history, not the current state. Section 8 verifies the release, the cohort, and the deterministic annual filing-year coverage restriction proposed over it.

## What this notebook can establish

Given the canonical artifacts present locally, it re-hashes the v5 packet manifest and its output JSONL files, checks the cohort ledger, verifies the full-cohort selection binding, inventories the screen-run directories, and verifies the named failed parent run's receipt counters and archive digest. It reproduces that *verification result* deterministically.

It cannot promise byte-identical results from a future live model call: provider behaviour is external. A future run must still use the governed CLI, a fresh run id, and a separately minted authorization.

One check is opt-in. `VERIFY_REUSABLE_PREFIX` is `False` by default, and while it is false this notebook reports **no** revalidated prefix partition, because it has not done the work that would establish one.

Section 8 adds three further read-only checks over artifacts that did not exist when this notebook was first written: the authoritative screen release and the candidate cohort derived from it, the annual filing-year coverage of that cohort, and the eligibility rule proposed over it. Those cells compute a partition and print it. They materialize nothing.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from the repository or one of its subdirectories.')

REPO_ROOT = find_repo_root()
print(f'Repository: {REPO_ROOT}')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def jsonl_row_count(path: Path) -> int:
    with path.open('rb') as handle:
        return sum(1 for _ in handle)


## 1. Repository identity and safe mode

A dirty worktree is reported rather than repaired. This notebook never stages, commits, pushes, or mutates repository files.

In [ ]:
def git(*args: str) -> str:
    return subprocess.check_output(['git', *args], cwd=REPO_ROOT, text=True).strip()

head = git('rev-parse', 'HEAD')
branch = git('branch', '--show-current')
status = git('status', '--short', '--untracked-files=all')
print(f'branch: {branch}')
print(f'HEAD:   {head}')
print('worktree: clean' if not status else f'worktree has changes:\n{status}')

EXECUTION_MODE = 'verify'  # This notebook intentionally supports only read-only verification.
assert EXECUTION_MODE == 'verify'

# Opt-in, local-only. When False (the default) the notebook performs fast
# artifact, hash and counter checks and nothing else. When True it additionally
# runs the committed continuation revalidation logic against the named parent
# archive: still no network, no credential, no model call, and no artifact
# written anywhere. It is slow because it reads the full packet corpus.
VERIFY_REUSABLE_PREFIX = False
print('VERIFY_REUSABLE_PREFIX:', VERIFY_REUSABLE_PREFIX)

## 2. Canonical v5 packet corpus

The v5 manifest is the current Stage 00 screening corpus authority. It binds the acquisition aggregate, shell and asset-backed determination inputs, Item 1 locator, output hashes, and cohort accounting.

In [ ]:
PACKET_RUN = REPO_ROOT / 'data/runs/baseline-packets/baseline-packets-domestic-text-lineage-v5-20260819'
PACKET_MANIFEST_PATH = PACKET_RUN / 'baseline_packet_manifest.json'
EXPECTED_PACKET_MANIFEST_SHA256 = '516b7020c657a7b656880444e0f98479c1aa46dca80bda9a1beafd846d7d88d8'

assert PACKET_MANIFEST_PATH.is_file(), f'Missing canonical manifest: {PACKET_MANIFEST_PATH}'
assert sha256_file(PACKET_MANIFEST_PATH) == EXPECTED_PACKET_MANIFEST_SHA256
PACKET_MANIFEST = load_json(PACKET_MANIFEST_PATH)
COUNTS = PACKET_MANIFEST['counts']

print('manifest SHA-256:', sha256_file(PACKET_MANIFEST_PATH))
print('packet contract:', PACKET_MANIFEST['schema_versions']['baseline_packet_manifest_v5'])
print('Item 1 locator:', PACKET_MANIFEST['item_one_locator'])
print('aggregate SHA:', PACKET_MANIFEST['aggregate_manifest_sha256'])
print('shell SHA:', PACKET_MANIFEST['shell_determination_manifest_sha256'])
print('asset-backed SHA:', PACKET_MANIFEST['asset_backed_determination_manifest_sha256'])


In [ ]:
expected_ledger = {
    'planned_rows': 8718,
    'firms_excluded': 1146,
    'retained_rows': 7572,
    'packets_built': 7042,
    'packet_failures': 530,
    'shell_only_true': 795,
    'asset_backed_only_true': 351,
    'both_true': 0,
}
for key, expected in expected_ledger.items():
    assert COUNTS[key] == expected, (key, COUNTS[key], expected)
assert COUNTS['planned_rows'] == COUNTS['firms_excluded'] + COUNTS['retained_rows']
assert COUNTS['retained_rows'] == COUNTS['packets_built'] + COUNTS['packet_failures']
assert COUNTS['firms_excluded'] == (COUNTS['shell_only_true']
                                    + COUNTS['asset_backed_only_true']
                                    + COUNTS['both_true'])

for key in ('planned_rows', 'firms_excluded', 'retained_rows', 'packets_built', 'packet_failures'):
    print(f'{key:>18}: {COUNTS[key]:,}')
print('failure reasons:', COUNTS['failures_by_reason'])
print('passages total:', f"{COUNTS['passages_total']:,}")


## 3. Re-hash the two v5 outputs

This reads the packet and failure JSONL files but does not alter them. On a normal local disk the packet JSONL check is the slowest cell because it is roughly 500 MB.

In [ ]:
for filename, expected_sha in PACKET_MANIFEST['output_hashes'].items():
    path = PACKET_RUN / filename
    assert path.is_file(), f'Missing output: {path}'
    observed_sha = sha256_file(path)
    assert observed_sha == expected_sha, (filename, observed_sha, expected_sha)
    rows = jsonl_row_count(path)
    print(f'{filename}: {rows:,} rows; SHA-256 verified')

assert jsonl_row_count(PACKET_RUN / 'universe_baseline_packets.jsonl') == COUNTS['packets_built']
assert jsonl_row_count(PACKET_RUN / 'baseline_packet_failures.jsonl') == COUNTS['packet_failures']


## 4. Full-cohort screen selection

The current selection is `full_cohort`, not a sample. Its `rows` array is empty **by contract**, and that is not an empty cohort: under `full_cohort@1` the packet manifest is the row authority, so the 7,042 packets it lists are the selected rows. A run binds the selection by digest and then enumerates rows from the manifest.

In [ ]:
SELECTION_PATH = (REPO_ROOT
                  / 'data/runs/universe-screen-selections'
                  / 'universe-screen-full-selection-v1-20260820'
                  / 'universe_screen_selection.json')
EXPECTED_SELECTION_SHA256 = '010b13bdf87e4fc6649f303340c465db12a3f9fa352ac55f70c000079fd8e7d4'

assert SELECTION_PATH.is_file(), f'Missing selection: {SELECTION_PATH}'
assert sha256_file(SELECTION_PATH) == EXPECTED_SELECTION_SHA256
SELECTION = load_json(SELECTION_PATH)

assert SELECTION['selection_kind'] == 'full_cohort'
assert SELECTION['sampling']['algorithm'] == 'full_cohort@1'
assert SELECTION['sampling']['seed'] is None
# The cohort binding is what makes this selection meaningful.
assert SELECTION['packet_manifest_sha256'] == EXPECTED_PACKET_MANIFEST_SHA256
# rows == [] is the contract, and packets_total is where the cohort size lives.
assert SELECTION['rows'] == []
assert SELECTION['counts']['packets_total'] == COUNTS['packets_built'] == 7042

print('selection SHA-256:', sha256_file(SELECTION_PATH))
print('selection kind:', SELECTION['selection_kind'])
print('enumerated rows in artifact:', len(SELECTION['rows']), '(empty by contract)')
print('cohort size from the bound packet manifest:', f"{SELECTION['counts']['packets_total']:,}")
print('binds canonical v5 manifest:', SELECTION['packet_manifest_sha256'] == EXPECTED_PACKET_MANIFEST_SHA256)

## 4b. The current high-recall prompt

All three current screen routes render the same committed V5 prompt. Its bytes are bound by digest in every authorization, so a run under a superseded prompt cannot start.

In [ ]:
SCREEN_PROMPT_PATH = REPO_ROOT / 'prompts/discovery/universe_high_recall_screen.v5.md'
EXPECTED_SCREEN_PROMPT_SHA256 = 'fee42d939f9eab590fdcbf055e7b2039e8a33a410dfc12257a47291d7a77d558'

assert SCREEN_PROMPT_PATH.is_file(), f'Missing prompt: {SCREEN_PROMPT_PATH}'
assert sha256_file(SCREEN_PROMPT_PATH) == EXPECTED_SCREEN_PROMPT_SHA256

print('V5 prompt SHA-256 verified:', sha256_file(SCREEN_PROMPT_PATH))
print('prompt path:', SCREEN_PROMPT_PATH.relative_to(REPO_ROOT))

## 5. Screen-run inventory

Every screen-run directory is either receipt-bearing or manifest-bearing, never both. A receipt-bearing directory is calibration or partial evidence only: it is non-authoritative by construction and must never become a classifier input or a release.

In [ ]:
SCREEN_RUNS_ROOT = REPO_ROOT / 'data/runs/universe-screens'
AUTHORITATIVE_MANIFEST_NAME = 'universe_screen_manifest.json'

receipt_bearing, manifest_bearing = [], []
for directory in sorted(path for path in SCREEN_RUNS_ROOT.iterdir() if path.is_dir()):
    receipt_path = directory / 'universe_screen_failure_receipt.json'
    manifest_path = directory / AUTHORITATIVE_MANIFEST_NAME
    if receipt_path.is_file():
        receipt = load_json(receipt_path)
        assert not manifest_path.exists(), f'receipt and manifest coexist: {directory}'
        receipt_bearing.append(directory.name)
        print(f"{directory.name}: receipt={receipt['reason_code']}, "
              f"stopped at row {receipt['stopping_row_index']:,}, authoritative=False")
    elif manifest_path.is_file():
        manifest_bearing.append(directory.name)
        print(f'{directory.name}: authoritative manifest present')
    else:
        print(f'{directory.name}: neither receipt nor manifest')

print()
print(f'receipt-bearing runs:  {len(receipt_bearing)}')
print(f'manifest-bearing runs: {len(manifest_bearing)}')
# The headline fact of the current state, asserted rather than described.
assert not manifest_bearing, 'an authoritative screen manifest exists; this notebook is stale'
print('No authoritative high-recall SCREEN release exists yet.')

## 6. The named failed parent run

`universe-high-recall-full-v4-20260821` is the run ADR-118 exists to continue. It completed 3,939 of the 7,042 rows and stopped at row 3,940 when a `countTokens` call timed out after 300 seconds — a call that route sends exactly once.

It holds a receipt and a raw-response archive, and nothing else. It is **immutable and permanently non-authoritative**: the authoritative and promotion loaders refuse it because a receipt is present, and no state change can clear that. ADR-118 may reuse its evidence only through a fresh continuation run that re-derives every reused row locally against the packet corpus and the unchanged strict validator. Being previously accepted by the parent counts for nothing.

In [ ]:
PARENT_RUN = REPO_ROOT / 'data/runs/universe-screens/universe-high-recall-full-v4-20260821'
PARENT_RECEIPT_PATH = PARENT_RUN / 'universe_screen_failure_receipt.json'
PARENT_ARCHIVE_PATH = PARENT_RUN / 'universe_screen_raw_responses.jsonl'
EXPECTED_PARENT_ARCHIVE_SHA256 = '08679414440968d9cbb77227fe0d6584803b9841232a6473620d482ad9078c34'

assert PARENT_RUN.is_dir(), f'Missing parent run: {PARENT_RUN}'
PARENT_RECEIPT = load_json(PARENT_RECEIPT_PATH)

assert PARENT_RECEIPT['stopping_row_index'] == 3940
assert PARENT_RECEIPT['records_completed_before_failure'] == 3939
assert PARENT_RECEIPT['raw_responses_captured'] == 3939
assert PARENT_RECEIPT['reason_code'] == 'provider_error'
assert 'provider_timeout' in PARENT_RECEIPT['detail']

assert sha256_file(PARENT_ARCHIVE_PATH) == EXPECTED_PARENT_ARCHIVE_SHA256
assert jsonl_row_count(PARENT_ARCHIVE_PATH) == PARENT_RECEIPT['records_completed_before_failure']

# What a failed run does not have is as important as what it has.
for absent in ('universe_screen_records.jsonl',
               'universe_screen_capture_ledger.jsonl',
               'universe_screen_manifest.json'):
    assert not (PARENT_RUN / absent).exists(), f'unexpectedly present: {absent}'

print('parent run:', PARENT_RUN.name)
print('reason:', PARENT_RECEIPT['reason_code'], '|', PARENT_RECEIPT['detail'])
print('completed rows:', f"{PARENT_RECEIPT['records_completed_before_failure']:,}",
      'of', f"{COUNTS['packets_built']:,}")
print('stopping row:', f"{PARENT_RECEIPT['stopping_row_index']:,}",
      f"(cik {PARENT_RECEIPT['stopping_cik']})")
print('archive SHA-256 verified:', sha256_file(PARENT_ARCHIVE_PATH))
print('records JSONL / capture ledger / authoritative manifest: all absent, as required')
print('remaining suffix to model-call:',
      f"{COUNTS['packets_built'] - PARENT_RECEIPT['records_completed_before_failure']:,}")

## 7. Optional local revalidation of the reusable prefix

This is the only expensive check here, and it is off by default. When `VERIFY_REUSABLE_PREFIX` is `True` the cell calls the **committed** continuation logic — `load_continuation_source` and `revalidate_source_prefix` — against the named parent archive. That code re-renders each prompt from its packet, resolves the passage references, and runs the unchanged strict validator, exactly as a continuation run would before sending anything.

The model route recorded on a reused row is **derived from the parent's hash-bound authorization**, never asserted here. The cell names that grant explicitly, re-hashes it, requires the digest the parent receipt itself recorded, and then reads `model_route` out of those verified bytes. A notebook that typed a provider and model name instead would be inventing provenance for rows it did not produce, which is precisely what the receipt-to-grant chain exists to prevent.

It makes no network, credential or model call and writes no artifact anywhere. It is slow because it reads the full packet corpus.

While the switch is `False`, this notebook reports no prefix partition at all. A partition is a measurement, and an unmeasured number is not a result.

In [ ]:
PREFIX_PARTITION = None

if not VERIFY_REUSABLE_PREFIX:
    print('VERIFY_REUSABLE_PREFIX is False — local revalidation was not run.')
    print('No prefix partition is reported, because none has been computed here.')
    print('Set VERIFY_REUSABLE_PREFIX = True above to compute it locally (slow, offline).')
else:
    import sys
    if str(REPO_ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / 'src'))
    from dynamic_ai_products.lineage_screen_continuation import (
        load_continuation_source, revalidate_source_prefix)
    from dynamic_ai_products.universe.lineage_screen import load_packet_run

    # The parent's own grant, named explicitly and pinned twice: once by the
    # digest expected here, and once by the digest the parent receipt recorded
    # when it ran. The route below is read out of those verified bytes.
    PARENT_AUTHORIZATION_PATH = (
        REPO_ROOT
        / 'data/runs/universe-screen-governance'
        / 'universe-screen-full-governance-v3-20260821'
        / 'screen_live_authorization.json')
    EXPECTED_PARENT_AUTHORIZATION_SHA256 = (
        '38d1eb078c508334f320ed20ac2fa13c945659d82da1e076eb304f4acfed1117')

    assert PARENT_AUTHORIZATION_PATH.is_file(), (
        f'Missing parent authorization: {PARENT_AUTHORIZATION_PATH}')
    parent_authorization_sha256 = sha256_file(PARENT_AUTHORIZATION_PATH)
    assert parent_authorization_sha256 == EXPECTED_PARENT_AUTHORIZATION_SHA256
    # The chain that makes this the parent's route rather than a guess.
    assert parent_authorization_sha256 == PARENT_RECEIPT['authorization_sha256']

    PARENT_AUTHORIZATION = load_json(PARENT_AUTHORIZATION_PATH)
    PARENT_MODEL_ROUTE = dict(PARENT_AUTHORIZATION['model_route'])
    print('parent authorization SHA-256 verified:', parent_authorization_sha256)
    print('matches receipt authorization_sha256:  True')
    print('model route derived from the grant:', PARENT_MODEL_ROUTE)

    prefix = load_continuation_source(
        PARENT_RUN, source_receipt_sha256=sha256_file(PARENT_RECEIPT_PATH))
    packets = list(load_packet_run(REPO_ROOT, PACKET_MANIFEST_PATH).packets)
    records, rejected = revalidate_source_prefix(
        prefix,
        packets=packets,
        prompt_text=SCREEN_PROMPT_PATH.read_text(encoding='utf-8'),
        model_route=PARENT_MODEL_ROUTE,
    )
    screened = sum(1 for r in records if r['record_kind'] == 'screened_packet')
    unverified = sum(1 for r in records if r['record_kind'] == 'model_evidence_unverified')
    PREFIX_PARTITION = {'rows': len(records), 'screened_packet': screened,
                        'model_evidence_unverified': unverified,
                        'by_reason': {k: v for k, v in rejected.items() if v}}
    assert screened + unverified == len(records) == PARENT_RECEIPT['records_completed_before_failure']
    print('locally revalidated prefix partition (computed just now):')
    for key, value in PREFIX_PARTITION.items():
        print(f'  {key}: {value}')
    print('No artifact was written and no external call was made.')

## 8. The release, the cohort, and the proposed coverage restriction

Three things are established here, in order, and none of them writes anything.

**The high-recall chain is finished.** `universe-screen-release-v1-20260823` is the
authoritative release over the 7,042-packet cohort. The human-review overlay adjudicated the
rows the screen could not resolve, and the classifier candidate cohort of **4,045 firms** was
built from the two. Every one of those artifacts is immutable and byte-unchanged; the cells
below read them and re-hash them, and change nothing.

**Annual filing-year coverage is a separate, later question.** For panel work each firm needs
an observation in each year of the window. That is a fact about filing dates, not about any
firm's business, and it is answered from the FRAME annual-filer inventory without a model.

**The proposed rule (ADR-138).** Require an annual filing in each calendar filing year 2022,
2023, 2024 and 2025. Record a 2021 filing but never require it. Let no filing after 2025 bear
on eligibility. Both FRAME annual-filer outputs count as coverage — domestic annual reports
and the foreign-private-issuer extension forms — because a firm filing 20-F is filing the
annual report its regime requires.

This restriction runs **after** the historical high-recall invocation, not before it. The
screen saw all 4,045 firms. The rule keeps a subset of them for analysis and settles no firm's
membership in any universe.

In [ ]:
# Read-only: the authoritative release, the overlay, and the candidate cohort.
RELEASE_MANIFEST = (REPO_ROOT / 'data/runs/universe-screen-releases'
                    / 'universe-screen-release-v1-20260823'
                    / 'universe_screen_release_manifest.json')
OVERLAY_MANIFEST = (REPO_ROOT / 'data/runs/universe-screen-human-review-overlays'
                    / 'universe-screen-v1-human-review-overlay-v2-20260824'
                    / 'universe_human_review_overlay_manifest.json')
COHORT_DIR = (REPO_ROOT / 'data/runs/universe-classifier-candidate-cohorts'
              / 'universe-classifier-candidate-cohort-v1-20260824')
COHORT_MANIFEST = COHORT_DIR / 'universe_classifier_candidate_cohort_manifest.json'

HAVE_RELEASE_CHAIN = all(p.is_file() for p in
                         (RELEASE_MANIFEST, OVERLAY_MANIFEST, COHORT_MANIFEST))
if not HAVE_RELEASE_CHAIN:
    print('The release chain is absent from this checkout; section 8 reports nothing.')
else:
    release = json.loads(RELEASE_MANIFEST.read_text(encoding='utf-8'))
    cohort = json.loads(COHORT_MANIFEST.read_text(encoding='utf-8'))
    print('authoritative high-recall release')
    print(f"  release_id      : {release['release_id']}")
    print(f"  manifest sha256 : {sha256_file(RELEASE_MANIFEST)}")
    print(f"  packets covered : {release['counts']['cohort_rows']}")
    print(f"  valid screened  : {release['counts']['valid_screened_rows']}")
    print()
    print('classifier candidate cohort (built from the release + overlay)')
    print(f"  cohort_id       : {cohort['cohort_id']}")
    print(f"  manifest sha256 : {sha256_file(COHORT_MANIFEST)}")
    print(f"  cohort rows     : {cohort['counts']['cohort_rows']}")
    print(f"  by origin       : model_screen={cohort['counts']['model_screen_admitted']}"
          f" human_review={cohort['counts']['human_review_admitted']}")
    print(f"  by status       : {cohort['counts']['by_screen_status']}")
    print()
    print('Every artifact above is immutable and is only read here.')

In [ ]:
# Read-only: annual filing-year coverage of the 4,045 cohort, from FRAME.
FRAME_DIR = (REPO_ROOT / 'data/runs/frame-full'
             / 'frame-live-full-v11-2020q1-2026q2-20260815')
FRAME_MANIFEST = FRAME_DIR / 'filer_frame_manifest.json'
ANNUAL_FILES = ('historical_annual_filers.jsonl', 'fpi_extension_filers.jsonl')

COVERAGE_YEARS = None
if not (HAVE_RELEASE_CHAIN and FRAME_MANIFEST.is_file()):
    print('The cohort or the FRAME run is absent; the coverage table is not computed.')
else:
    frame = json.loads(FRAME_MANIFEST.read_text(encoding='utf-8'))
    # Each annual-filer file is read only after it re-hashes to the digest its own
    # manifest records, exactly as the governed builder does.
    for name in ANNUAL_FILES:
        observed = sha256_file(FRAME_DIR / name)
        recorded = frame['output_hashes'][name]
        assert observed == recorded, (name, observed, recorded)
        print(f'  {name}: re-hashes to its manifest entry')

    cohort_rows = [json.loads(line) for line in
                   (COHORT_DIR / 'universe_classifier_candidate_records.jsonl')
                   .read_text(encoding='utf-8').splitlines() if line.strip()]
    years_by_cik: dict[str, set] = {}
    for name in ANNUAL_FILES:
        for line in (FRAME_DIR / name).read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            years_by_cik.setdefault(row['cik'], set()).add(int(row['filing_date'][:4]))
    COVERAGE_YEARS = {r['cik']: years_by_cik.get(r['cik'], set()) for r in cohort_rows}

    print()
    print(f'annual filing-year coverage of the {len(cohort_rows)} cohort firms')
    print('  year   firms with an annual filing   share')
    for year in range(2020, 2027):
        present = sum(year in y for y in COVERAGE_YEARS.values())
        share = present / len(COVERAGE_YEARS)
        print(f'  {year}   {present:>27}   {share:6.1%}')

In [ ]:
# Read-only: the proposed eligibility rule and its decomposition.
REQUIRED_YEARS = (2022, 2023, 2024, 2025)
OPTIONAL_YEAR = 2021

if COVERAGE_YEARS is None:
    print('Coverage was not computed; the rule is not evaluated.')
else:
    complete, without_2021, excluded = [], [], []
    for cik, observed in COVERAGE_YEARS.items():
        if set(REQUIRED_YEARS) <= observed:
            (complete if OPTIONAL_YEAR in observed else without_2021).append(cik)
        else:
            excluded.append(cik)
    included = len(complete) + len(without_2021)

    print('proposed rule: an annual filing in each of 2022, 2023, 2024 and 2025')
    print('  2021 is recorded and never required; no filing after 2025 counts')
    print()
    print(f'  source cohort                    : {len(COVERAGE_YEARS)}')
    print(f'  included                         : {included}')
    print(f'    complete_2021_2025             : {len(complete)}')
    print(f'    2021_missing_2022_2025_present : {len(without_2021)}')
    print(f'  excluded                         : {len(excluded)}')
    print()
    missing_counts = {n: 0 for n in range(1, 5)}
    for cik in excluded:
        gaps = sum(y not in COVERAGE_YEARS[cik] for y in REQUIRED_YEARS)
        missing_counts[gaps] += 1
    print('  excluded firms by number of missing required years:')
    for n, count in missing_counts.items():
        print(f'    missing {n} year(s): {count}')
    assert included + len(excluded) == len(COVERAGE_YEARS), 'the partition must be total'
    print()
    print('  the two populations partition the cohort exactly; no firm is dropped silently')

### What section 8 did and did not do

**Did:** read four immutable manifests, re-hash both FRAME annual-filer files against the
digests their own manifest records, and compute a partition of the 4,045 candidate firms into
those with complete required-year coverage and those without.

**Did not:** call a model, reach SEC, mint governance, create a run directory, or write a byte.
The eligibility cohort is *not* materialized here. Producing it is a separate governed step —
`--mode build-annual-coverage-cohort`, which pins every input by digest, validates all three
output contracts before writing, and writes once.

**The existing high-recall artifacts are unchanged.** The screen release, the human-review
overlay and the candidate cohort are byte-identical to what the governed runs produced. This
restriction reads them and would write elsewhere. It is a *later* deterministic filter, applied
after the historical high-recall invocation and never a rescoping of it: the screen saw every
one of these firms.

**What the resulting cohort is, and is not.** It is an analysis-eligibility cohort. It is not a
software universe, not a classifier result, and it settles no firm's membership. Requiring a
filing in every year selects a survivor / continuing-reporter sample: firms acquired, taken
private, deregistered, delisted or failed inside the window are dropped, as are firms that first
registered after it opened. Any estimate computed on it is conditional on surviving as a
reporting registrant.

## 9. What comes next

The required sequence is below. It is deliberately rendered as an ordered list of governed steps rather than an executable command line: this notebook does not run the pipeline, and a copy-pasteable command in a verification notebook invites exactly the accident the governance model exists to prevent.

Each step is a separate approval. Nothing about reading this notebook authorizes any of them.

In [ ]:
# The sequence below is current as of ADR-138. The continuation, the release audit
# and the candidate cohort are done; they are listed as completed rather than
# deleted, so a reader can see where the chain now stands.
COMPLETED_SO_FAR = [
    'Governed continuation over the remaining suffix, reusing the named failed run\'s '
    'archived prefix byte for byte.',
    'High-recall release audit and the human-review overlay over the rows the screen '
    'could not resolve.',
    'Freeze of the authoritative release universe-screen-release-v1-20260823.',
    'The immutable 4,045-row classifier candidate cohort built from the release and '
    'the overlay.',
]

NEXT_REQUIRED_SEQUENCE = [
    'Materialize the ADR-138 annual filing-year coverage cohort under its own governed '
    'CLI mode, pinning the candidate cohort and both FRAME annual-filer inputs by '
    'digest. Deterministic and model-free; it writes an inclusion artifact and a '
    'separate exclusion artifact and alters no existing artifact.',
    'Decide, as a separate entry, which cohort the classifier runs over. Creating the '
    'eligibility cohort does not choose it.',
    'Complete the classifier work: the firm-level pilot, then the successor the '
    'calibration evidence supports.',
    'Derive deterministic tiers and freeze UNIVERSE_v1.',
    'Only then decide the PCT sample or census and Dev30 instrument validation.',
]

print('Completed — verified by the artifacts this notebook reads:')
for position, step in enumerate(COMPLETED_SO_FAR, start=1):
    print(f'  {position}. {step}')
print()
print('Next required sequence — described, not executed:')
for position, step in enumerate(NEXT_REQUIRED_SEQUENCE, start=1):
    print(f'  {position}. {step}')

## Reproducibility boundary

A researcher with this checkout and the same immutable artifacts can reproduce every check in this notebook, including the optional local revalidation. A future model run is reproducible as a governed experiment — exact inputs, route, prompt, authorization, captures and outputs are recorded — but is not assumed byte-identical merely because an external provider is called again.

Nothing in this notebook is an authority. For any run claim, read the named manifest and its hashes first, then the committed source, schemas, tests and ADRs at this revision.